The `jax.tools.colab_tpu.setup_tpu()` is deprecated. Modern TPUs like **v5e** should natively run on a TPU VM architecture.

**I'm using TPU v5e**.

In [11]:
# We'll need to uninstall existing JAX, JAXlib, and related libraries to ensure a clean slate.
!pip uninstall -y jax jaxlib ml-dtypes flax optax orbax-checkpoint

# Then, install all desired packages (JAX, JAXlib, ml-dtypes, Flax, Optax) in a single command.
# This ensures that pip uses the custom JAX index and find-links for all dependency resolutions, preventing JAX from being downgraded.
# This compatible set works: JAX 0.5.3, Flax 0.10.0, Optax 0.2.8
# JAX 0.5.3 fulfills the requirement of optax 0.2.8 (jax>=0.5.3) while Flax 0.10.0 is compatible with JAX 0.5.3 and addresses the 'MainTrace' import error.
!pip install --upgrade \
  "jax[tpu]==0.5.3" \
  "jaxlib==0.5.3" \
  "ml-dtypes==0.5.0" \
  flax==0.10.0 \
  optax==0.2.8 \
  -i https://us-python.pkg.dev/ml-oss-artifacts-published/jax/simple/ \
  -f https://storage.googleapis.com/jax-releases/libtpu_releases.html

In [2]:
import jax
import jax.numpy as jnp
from flax import nnx
import optax
import numpy as np
import urllib.request

print(jax.devices())
print(jax.device_count())

/usr/local/lib/python3.13/dist-packages/jax/_src/cloud_tpu_init.py:82: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


[TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]
1


In [3]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = urllib.request.urlopen(url).read().decode('utf-8')

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {char: i for i,char in enumerate(chars)}
itos = { i:ch for i,ch in enumerate(chars) }

data = np.array([stoi[c] for c in text], dtype=np.int32)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

block_size = 64
batch_size = 32

def get_batch(split):
  data_split = train_data if split == "train" else val_data

  # random starting points
  ix = np.random.randint(0, len(data_split) - block_size, size=(batch_size,))
  x = np.stack([data_split[i : i+block_size] for i in ix])
  y = np.stack([data_split[i+1 : i+block_size+1] for i in ix])

  return jnp.array(x), jnp.array(y)


In Flax NNX, **models** are standard Python classes while **layers** are defined inside an `__init__` constructor --> similar to PyTorch.

With Flax Linen (previous way of building models) models were data classes while layers were defined inside a `setup`.



Flax NNX uses the `nnx.Rngs` class to simplify Jax's handling of random states.

## `__call__`

### Training Mode
During training, models usually use the entire input sequence at once. We use `decode=False` to tell the attention mechanism that it can attend to all elements in the input sequence since we don't need to simulate a sequential generation process.

### Inference Mode
When generating a sequence token by token, we set `decode=True`. The attention mechanism will only attend to previously generated tokens (and the current token being predicted) to prevent information leakage from future tokens.

In [4]:
class TransformerBlock(nnx.Module):
  def __init__(self, n_embd, n_heads, rngs: nnx.Rngs):
    self.attention = nnx.MultiHeadAttention(num_heads=n_heads, in_features=n_embd, rngs=rngs)
    self.ln1 = nnx.LayerNorm(n_embd, rngs=rngs)
    self.ln2 = nnx.LayerNorm(n_embd, rngs=rngs)
    self.ff1 = nnx.Linear(n_embd, n_embd * 4, rngs=rngs)
    self.ff2 = nnx.Linear(n_embd * 4, n_embd, rngs=rngs)

  def __call__(self, x, mask):
    x = x + self.attention(self.ln1(x), mask=mask, decode=False)
    x = x + self.ff2(jax.nn.relu(self.ff1(self.ln2(x))))
    return x

In [5]:
class NNXLanguageModel(nnx.Module):
  def __init__(self, vocab_size, n_embd=64, n_heads=4, block_size=64, rngs: nnx.Rngs = None):
    self.token_emb = nnx.Embed(vocab_size, n_embd, rngs=rngs)
    self.positional_emb = nnx.Embed(block_size, n_embd, rngs=rngs)
    self.block = TransformerBlock(n_embd, n_heads, rngs=rngs)
    self.ln_final = nnx.LayerNorm(n_embd, rngs=rngs)
    self.language_model_head = nnx.Linear(n_embd, vocab_size, rngs=rngs)

  def __call__(self, x):
    batch_size, seq_len = x.shape

    # causal mask is a triangular matrix used in the self-attention layers of an autoregressive transformer model
    # it prevents tokens from attending to future positions in a sequence
    mask = nnx.make_causal_mask(x)

    token_emb = self.token_emb(x)
    positional_emb = self.positional_emb(jnp.arange(seq_len)[None, :])
    x = token_emb + positional_emb

    x = self.block(x, mask=mask)

    return self.language_model_head(self.ln_final(x))

`softmax_cross_entropy_with_integer_labels` takes the model's raw scores and the ground truth (**targets**), converts the logits to probabilities, and then calculates the difference between these probabilities and the true lables using cross-entropy.

Cross-entropy loss measures how well your predicted probability distribution matches the true distribution.

## Training

In [6]:
model = NNXLanguageModel(vocab_size=vocab_size, rngs=nnx.Rngs(0))
optimizer = nnx.Optimizer(model, optax.adamw(learning_rate=1e-3))

@nnx.jit
def train_step(model, optimizer, inputs, targets):
  def loss_fun(model):
    logits = model(inputs) # raw, unnormalized prediction scores
    loss = optax.softmax_cross_entropy_with_integer_labels(logits=logits, labels=targets).mean()
    return loss

  loss, grads = nnx.value_and_grad(loss_fun)(model)
  optimizer.update(grads)

  return loss

## Validation

In [7]:
@nnx.jit
def eval_step(model, inputs, targets):
  logits = model(inputs)
  loss = optax.softmax_cross_entropy_with_integer_labels(logits=logits, labels=targets).mean()
  return loss

def estimate_loss(model, eval_iters=100):
  out = {}
  model.eval()
  for split in ["train", "val"]:
    losses = jnp.zeros(eval_iters)
    for k in range(eval_iters):
      xb, yb = get_batch(split)
      loss = eval_step(model, xb, yb)
      losses = losses.at[k].set(loss)
    out[split] = losses.mean().item()

  model.train()
  return out

In [8]:
eval_interval = 200

for i in range(1000):
  if i % eval_interval == 0 or i == 999:
    losses = estimate_loss(model)
    print(f"step {i}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

  x_batch, y_batch = get_batch("train")
  loss = train_step(model, optimizer, x_batch, y_batch)

step 0: train loss 4.7528, val loss 4.7369
step 200: train loss 2.5236, val loss 2.5336
step 400: train loss 2.3892, val loss 2.4005
step 600: train loss 2.2085, val loss 2.2469
step 800: train loss 2.0809, val loss 2.1319
step 999: train loss 1.9947, val loss 2.0748


## Generation

In [9]:
from jax._src.numpy.lax_numpy import block
def generate_text(model, max_new_tokens, context_window=64):
  context = jnp.array([[stoi["\n"]]])
  rng = jax.random.PRNGKey(42)

  for _ in range(max_new_tokens):
    idx_cond = context[:, -context_window:] # all rows, last 64 columns
    logits = model(idx_cond)
    logits_last = logits[:, -1, :]

    rng, subkey = jax.random.split(rng)
    next_id = jax.random.categorical(subkey, logits_last)

    context = jnp.concatenate([context, next_id[:, None]], axis=1)

  generated_ids = context[0].tolist()
  return "".join([itos[i] for i in generated_ids])

In [10]:
print(generate_text(model, max_new_tokens=500, context_window=block_size))



O.

OVINGLBEY:
Whic,-
Denvy you; me, gone not me amy lorted
With wordeaven therear cesn.
Gram, thest by othiss grequidy.


LUCEENTIZENTIO:
Why shy tromen wrommorer sholds wrosciught
MaRT Sorsch lhlagant.

COMLTAMBUGESTEOL:
Wely geast dam ands:
But I gam; and ir theimp, thon be your his her shatrrind raikreigh.

For:
Whank the and them hesect.

PUCETLER:
Beadray, as from noth a not guld, of bead monter
Is notivess; han day Cuveseart Vikes.

Stembalotily gu old an the ray rea inscomst a with-dand 


# Building LLMs from Scratch

LLMs are like any other machine learning algorithm.

## 1. Prep the Data
Garbage in versus garbage out. Any ML, data science, or analytical project starts with manageable data.

With LLMs in particular you'll need to:
* **Collect the corpus:** gather all the raw text data your model will learn from.
* **Build a vocabulary:** identify all unique tokens (characters, subwords, or words) in the dataset in order to determine the `vocab_size`
* **Create token mappings:** you're making a lookup table that converts human-readable strings to integers and intergers back to strings.
* **Encode and split:** convert the entire text corpus into a 1D array of integers. Split that into a training set and a validation set.



## 2. Define the Context
* Set the Batch Size ($B$) for parallel processing and the context window/sequence length ($T$) for the model's memory.
* Randomly sample the encoded data to create input sequences ($x$) of length $T$.
* For every input sequence $x$, create a target seqence $y$ shifted one timestep into the future.


## 3. Create the Model's Architecture

* **Token embeddings:** are a lookup table that convert integer IDs into trainable, dense mathematical vectors.
* **Positional embeddings:** are a secondary vector added to the token embeddings to explicitly encode sequence order. This bypasses the transformer's inherent lack of sequential awareness. The positional embeddings add two pieces of information for each token:
  * Semantic meaning
  * Its position in the sequence
* **Self-Attention with Causal Masking:** this is the core mechanism where tokens communicate with one another to build context. A **causal mask** is applied to prevent tokens from *cheating* by looking at future positions.
* **Feed-Forward networks:** this is the standard neural network layers that independently process the context gathered by the attention mechanism.
* **Lanuage Model Head:** this is the final layer that projects the internal vector representation back out to the `vocab_size`. It outputs raw prediction scores (aka logits).


## 4. The Training Loop
* You'll need to initialize an optimizer like **AdamW** to govern how weights are updated based on errors.
* The **forward pass** passes a batch of $x$ through the architecture to generate logits.
* You'll use a function like **cross-entropy** to measure the distance between the model's predictions and the actual target tokens $y$.
  * You're essentially calculating the loss.
* For the **backward pass**, you'll calculate the gradients to determine how much each indivdual weight contributed to the total loss
* The **optimizer step** adjusts the model's weights in the direction that minimizes the loss.


## 5. Validation and Evaluation
* You'll preiodically run batches from the validation split through a forward pass to calculate **validation loss**.
* Track the gap between training loss and validation loss.
  * Overfitting occurs when the **training loss** drops while the **validation loss** plateaus or climbs. The model is *memorizing* the training sequences instead of *learning* the underlying structure of the language.


## 6. Autoregressive Inferencing (Generation)
* Provide a starting token (like a newline character or a user prompt) converted into an integer ID.
* Run a forward pass, extract the logits, convert the logits into probabilities, and sample the next token ID.
* Add the newly generated token to the running input sequence, drop the oldest token if you exceed the context window, and feed the new sequence back into the model to predict the next token.